In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


data = pd.read_csv(
    "./../data/raw/shopify_sales_data_v2.csv",
    dtype={"sku": str, "art_number": str},
)
for col in ["sold_at", "listed_at", "published_at"]:
    data[col] = pd.to_datetime(data[col], errors="coerce", utc=True)

print(len(data))

In [ ]:
data.head()

In [ ]:
# Checking coverage

for col in ["condition_grade", "art_number", "size", "colour", "brand", "measurements"]:
    filled = data[col].notna() & (data[col].astype(str).str.strip() != "")
    print(f"{col:18s} {filled.mean():6.1%}")

In [ ]:
no_prod = data["brand"].isna() | (data["brand"].astype(str).str.strip() == "")
print(no_prod.mean())                     # should be ~30.3%
print(data.loc[no_prod, "sold_at"].min(), "→", data.loc[no_prod, "sold_at"].max())
print(data.loc[no_prod, "title"].head(10))

In [ ]:
CORE_COLS = ["brand", "size", "colour", "condition_grade",
             "list_price", "listed_at", "published_at",
             "sold_at", "sold_price", "title", "sku"]

filled = pd.DataFrame({
    col: data[col].notna() & (data[col].astype(str).str.strip() != "")
    for col in CORE_COLS
})
complete = filled.all(axis=1)

print(f"complete (all core fields): {complete.sum()} rows ({complete.mean():.1%})")
print(f"of which also have measurements: "
      f"{(complete & (data['measurements'].fillna('') != '')).sum()}")

# What's blocking the incomplete rows — count missing fields per column:
print("\nmissing-field counts among incomplete rows:")
print((~filled[~complete]).sum().sort_values(ascending=False))

In [ ]:
# Checking range

CATEGORICAL_COLS = ["brand", "size", "colour", "condition_grade", "season", "product_name"]

for col in CATEGORICAL_COLS:
    vc = data[col].fillna("").astype(str).str.strip().value_counts()
    n_unique = len(vc)
    less_than_5 = (vc < 5).sum()
    print(f"\n=== {col} — {n_unique} unique values, {less_than_5} appear less than five times ===")
    print(vc.head(20))
    if n_unique > 20:
        print("  ... tail:", ", ".join(vc.tail(10).index))

In [ ]:
spike = data["listed_at"].dt.date.value_counts().head(3)
print(spike)  # how many rows share the import date?
data["clock_start"] = data["listed_at"]
# rows on the spike date get clock_start = NaT (untrusted)

In [ ]:
price = data["sold_price"]
valid = price.notna() & (price > 0)          # log needs strictly positive
print(f"usable price rows: {valid.sum()} / {len(data)}")

In [ ]:
# --- summary ---
print(price.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2))
print("\nzero/negative:", (price <= 0).sum(), " | null:", price.isna().sum())
print("top 10 highest:", sorted(price.dropna(), reverse=True)[:10])

In [ ]:
# --- helper: price distribution by a category (top N groups by volume) ---
def price_by(col, topn=12, logy=True):
    d = data.loc[valid, [col, "sold_price"]].copy()
    d[col] = d[col].fillna("(missing)").astype(str).str.strip().replace("", "(missing)")
    keep = d[col].value_counts().head(topn).index
    d = d[d[col].isin(keep)]
    order = d.groupby(col)["sold_price"].median().sort_values().index  # order by median
    fig, ax = plt.subplots(figsize=(10, 5))
    d.boxplot(column="sold_price", by=col, ax=ax, rot=45,
              positions=range(len(order)), grid=False)
    ax.set_xticklabels(order)
    if logy: ax.set_yscale("log")
    ax.set_title(f"sold_price by {col}"); plt.suptitle(""); ax.set_xlabel("")
    plt.tight_layout(); plt.show()
    # median table (the numbers behind the plot)
    print(d.groupby(col)["sold_price"].agg(["count", "median"]).round(2).sort_values("median"))

price_by("brand")
price_by("size")

In [ ]:
# --- condition: is the free-text grade MONOTONIC with price? (decides ordinal vs nominal) ---
cond = (data.loc[valid]
        .groupby(data["condition_grade"].fillna("(missing)").str.strip())["sold_price"]
        .agg(["count", "median"])
        .sort_values("median"))
print(cond.round(2))   # if median rises cleanly across grades -> encode as ordered ordinal

In [ ]:
# --- discount signal: sold vs list price ---
m = valid & data["list_price"].notna() & (data["list_price"] > 0)
ratio = (data.loc[m, "sold_price"] / data.loc[m, "list_price"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ratio.clip(0, 2), bins=60)          # clip tail so the shape is visible
ax.axvline(1.0, color="k", ls="--", lw=1)   # sold == list
ax.set_title("sold / list price ratio"); ax.set_xlabel("ratio")
plt.tight_layout(); plt.show()

print(ratio.describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
print(f"sold BELOW list: {(ratio < 1).mean():.1%}   above: {(ratio > 1).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(price[valid], bins=60)
ax[0].set_title("sold_price (raw)"); ax[0].set_xlabel("£")

logp = np.log(price[valid])
ax[1].hist(logp, bins=60)
ax[1].set_title("log(sold_price)"); ax[1].set_xlabel("log £")
plt.tight_layout(); plt.show()

# how skewed, and does log fix it? (0 ≈ symmetric)
print("skew raw :", round(price[valid].skew(), 2))
print("skew log :", round(logp.skew(), 2))

In [ ]:
from collections import Counter
last = Counter(
    str(n).lower().split()[-1]
    for n in data["product_name"].dropna() if str(n).strip()
)
print("most common final words:", last.most_common(40))

In [ ]:
filled = data["product_type"].fillna("").str.strip() != ""
print(f"product_type filled: {filled.mean():.1%}   blank: {(~filled).sum()} rows")
print(data.loc[~filled, "product_name"].head(20))   # what's missing?

## EDA Conclusions

**Target & metric**
- Target: log(sold_price). Raw skew 1.22 → log skew -0.51.
- Metric: RMSLE / RMSE-in-log-space (percentage error).

**Cleaning rules (→ clean.py)**
- Normalise categoricals (brand, size, colour): merge misspellings
  (Stone Isand → Stone Island); collapse <5-example values into "other" or a relevant category.

**Feature engineering (→ features/)**
- product_type = shopify productType field or if empty: parse product_name fallback
- measurements → numeric (pit-to-pit, waist, inside leg).

**Feature shortlist (known at listing time)**
- Strong: brand, product_type, condition_grade, size, colour
- Test also: season, measurements

**Leakage — EXCLUDED (with reasons)**
- list_price — ≈ sold_price; target in disguise
- days-on-market — leakage (unknown until it sells)
- tags: sold-out-hidden (post-sale, on every row); SEO_NOW / staff-picks (operational).
- everything else not in the shortlist.

**Validation**
- Time-based split at 2025-12-14 (~80/20). No random k-fold.
- Justified: median sold_price drifts +~25-30% across the 2.5yr window.
